In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Configuración para que los gráficos se vean mejor
sns.set_theme(style="whitegrid")

In [ ]:
url = "https://raw.githubusercontent.com/sharmaroshan/Churn-Modelling-Dataset/master/Churn_Modelling.csv"
df = pd.read_csv(url)

# Guardar localmente en tu carpeta data
df.to_csv('../data/churn_raw.csv', index=False)
df.head()

In [ ]:
# Quitamos lo que no sirve para la IA
df_limpio = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Verificamos que no haya nulos
print(df_limpio.isnull().sum())

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_limpio, x='Age', hue='Exited', fill=True, common_norm=False)
plt.title('Distribución de Edad: Clientes Leales (0) vs Fugados (1)')
plt.show()

In [ ]:
# Filtramos solo las columnas que son números para poder calcular la correlación
columnas_numericas = df_limpio.select_dtypes(include=['int64', 'float64']).columns
matriz_corr = df_limpio[columnas_numericas].corr()

# Dibujamos el Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(matriz_corr, annot=True, cmap='RdYlGn', fmt=".2f", linewidths=0.5)
plt.title('Mapa de Calor de Correlaciones')
plt.show()

In [ ]:
# 1. Convertir Género a 0 y 1
df_limpio['Gender'] = df_limpio['Gender'].map({'Female': 0, 'Male': 1})

# 2. Convertir Países en columnas separadas (One-Hot Encoding)
df_final = pd.get_dummies(df_limpio, columns=['Geography'], drop_first=True)

# 3. Guardar el dataset procesado para la siguiente fase
df_final.to_csv('../data/churn_procesado.csv', index=False)

print("Transformación completada.")
print(df_final.head())

In [ ]:
from sklearn.model_selection import train_test_split

# Separamos características de la etiqueta
X = df_final.drop('Exited', axis=1)
y = df_final['Exited']

# Dividimos en 80% para entrenar y 20% para evaluar qué tan bueno es el modelo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Set de entrenamiento: {X_train.shape[0]} registros")
print(f"Set de prueba: {X_test.shape[0]} registros")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Creamos el modelo (con 100 árboles de decisión)
modelo_churn = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. ENTRENAMIENTO: Aquí la IA aprende los patrones de los 8000 registros
modelo_churn.fit(X_train, y_train)

# 3. PREDICCIÓN: Le pedimos que adivine el resultado de los 2000 que no conoce
y_pred = modelo_churn.predict(X_test)

print("¡Modelo entrenado y predicciones realizadas!")

In [ ]:
# Ver el porcentaje de acierto general
exactitud = accuracy_score(y_test, y_pred)
print(f"Exactitud del modelo: {exactitud:.2%}")

# Matriz de Confusión: ¿Dónde se equivoca?
print("\n--- Matriz de Confusión ---")
print(confusion_matrix(y_test, y_pred))

# Reporte detallado
print("\n--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred))

In [ ]:
# Extraer la importancia de las variables
importancias = pd.DataFrame({'Variable': X.columns, 'Importancia': modelo_churn.feature_importances_})
importancias = importancias.sort_values(ascending=False, by='Importancia')

# Graficar
plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Variable', data=importancias, palette='viridis')
plt.title('¿Qué variables pesan más para la Inteligencia Artificial?')
plt.show()

In [ ]:
import joblib

# Creamos la carpeta models si no existe (por si acaso)
import os
if not os.path.exists('../models'):
    os.makedirs('../models')

# Guardamos el modelo y las columnas (muy importante para Power BI después)
joblib.dump(modelo_churn, '../models/modelo_churn_final.pkl')
joblib.dump(X.columns.tolist(), '../models/columnas_modelo.pkl')

print("¡Modelo y metadatos guardados en la carpeta /models!")